#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim,col,length

#Reading From Bronze

In [0]:
df=spark.table("workspace.bronze.erp_cust_az12")

#Data Transformation

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df=df.withColumn(field.name,F.trim(col(field.name)));
        

##Customer ID Cleanup

In [0]:
df=df.withColumn("cid",
        F.when(col("cid").startswith("NAS"),F.substring(col("cid"),4,F.length(col("cid"))))
        .otherwise(col("cid"))
)

##Birthdate Validation


In [0]:
df=df.withColumn("bdate",
                 F.when(col("BDATE") > F.current_date(),None)
                 .otherwise(col("BDATE")))
                 


In [0]:
df.display()

##Normalization


In [0]:
df=df.withColumn("gen",
                 F.when(F.upper(col("GEN")).isin("MALE","M"),"Male")
                    .when(F.upper(col("GEN")).isin("FEMALE","F"),"Female")
                    .otherwise("n/a"))

##Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}

for old_name,new_name in RENAME_MAP.items():
    df=df.withColumnRenamed(old_name,new_name)

#Sanity check of Dataframe

In [0]:

df.limit(10).display()

#Writing Into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.erp_customers")

#Sanity check of Silver Table

In [0]:
%sql
Select * FROm silver.erp_customers
LIMIT 10;